In [24]:
import numpy as np
# import scipy
import matplotlib.pyplot as plt
import pandas as pd
# from pandas import DataFrame
# from astropy.coordinates import SkyCoord
# import time
# import seaborn as sns
# from PIL import Image
import os
# from tqdm import tqdm_notebook, trange

# import astropy
# from astropy.io import fits
# import astropy.units as u
# import astropy.constants as const
# # from astropy.table import Table as t
# from astropy.cosmology import FlatLambdaCDM
# cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

In [25]:
#%pip install panoptes-client

In [26]:
from panoptes_client import Panoptes, Workflow, Project, Subject, SubjectSet

In [27]:
import getpass
Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))

# Upload Dual Image Subjects (Cutouts)

In [28]:
# Get our subject set
#this is beta
subject_set_id = 137684 # this is our actual subject set
# subject_set_id_test = 127722 # this is our test subject set

subject_set = SubjectSet.find(subject_set_id)

# Galaxy Zoo project ID
gz_project_id = 30908

In [29]:
csv = pd.read_csv('/users/7/aimees/AI_Inspector/betatest/beta.csv') #/users/7/aimees/AI_Inspector/betatest/beta.csv
source_index = range(0, len(csv))
csv['Index'] = source_index
print(csv.columns)

Index(['source_fits', 'det_code', 'det_xmin', 'det_ymin', 'det_xmax',
       'det_ymax', 'det_centerx', 'det_centery', 'cutout_x0', 'cutout_y0',
       'cutout_x1', 'cutout_y1', 'cutout_centerx', 'cutout_centery',
       'local_xmin', 'local_ymin', 'local_xmax', 'local_ymax', 'local_centerx',
       'local_centery', 'zeroth_order', 'source_type', 'cutout_stem', 'sam_x0',
       'sam_y0', 'sam_x1', 'sam_y1', 'mask_h', 'mask_w', 'mask_encoded',
       'Index'],
      dtype='object')


In [30]:
manifest_columns = ['source_fits', 'det_code', 'det_xmin', 'det_ymin', 'det_xmax',
       'det_ymax', 'det_centerx', 'det_centery', 'cutout_x0', 'cutout_y0',
       'cutout_x1', 'cutout_y1', 'cutout_centerx', 'cutout_centery',
       'local_xmin', 'local_ymin', 'local_xmax', 'local_ymax', 'local_centerx',
       'local_centery', 'zeroth_order', 'source_type', 'cutout_stem', 'sam_x0',
       'sam_y0', 'sam_x1', 'sam_y1', 'mask_h', 'mask_w', 'mask_encoded',
       'Index']

In [31]:
# Creating a system that allows me to check what's been uploaded

# Create the file initially (if it doesn't exist)
filename = 'uploaded_subjects_by_id_field.txt'

# Make sure the file exists
open(filename, 'a').close()

def update_txt_file(id_field, subject_id=None):
    """Appends a new id_field (and its subject id, if known) to the tracking file.

    Stored as 'id_field,subject_id' so replacements can look up which subject to
    remove later; a replacement is recorded as a new line rather than an edit, and
    the most recent line for a given id_field wins when reading.
    """
    with open(filename, 'a') as f:
        f.write(f"{id_field},{subject_id if subject_id is not None else ''}\n")

def get_uploaded_id_map():
    """Returns {id_field: subject_id}, using the most recent line per id_field.

    subject_id is None for entries written before subject-id tracking was added.
    """
    id_map = {}
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',', 1)
            id_field = parts[0]
            subject_id = parts[1] if len(parts) > 1 and parts[1] else None
            id_map[id_field] = subject_id
    return id_map

def get_uploaded_ids():
    return np.array(list(get_uploaded_id_map().keys()))

def check_if_uploaded(id_field):
    """Checks if the given id_field is already in the tracking file."""
    try:
        return str(id_field) in get_uploaded_id_map()
    except FileNotFoundError:
        return False  # If the file doesn't exist yet, nothing has been uploaded

In [20]:
import functools
import os
import threading
from multiprocessing.pool import ThreadPool
from tqdm import tqdm
from tqdm import tqdm_notebook

def upload_selected_galaxies(selected_galaxies, subject_set_id, subject_png_save_dir, gz_project_id, manifest_columns, n_processes=10, batch_size=10, replace_ids=None):
    """
    Upload selected galaxies to Galaxy Zoo using threading (batching adds to subject set).

    replace_ids: optional iterable of `cutout_stem` values to force-upload even though
    they're already in the tracking file (e.g. corrected subjects). For each one, the
    previously-uploaded subject is removed from the subject set once its replacement
    has been added, provided that subject's id was recorded in the tracking file.
    """
    # Connect once globally
    client = Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))

    replace_ids = {str(x) for x in replace_ids} if replace_ids else set()

    try:
        project = Project.find(gz_project_id)
        subject_set = SubjectSet.find(subject_set_id)

        # Prepare manifest
        manifest = []
        uploaded_id_map = get_uploaded_id_map()

        for i in range(len(selected_galaxies)):
            row = selected_galaxies.iloc[i]
            id_field = str(row['cutout_stem'])
            is_replacement = id_field in replace_ids

            if id_field in uploaded_id_map and not is_replacement:
                continue

            entry = {
                'jpg_locs': [
                    subject_png_save_dir + 'sam_results/' + str(row['cutout_stem'] + '_result.jpg'),
                    subject_png_save_dir + 'sam_results_precontsub/' + str(row['cutout_stem'] + 'precont_AND_sam.jpg')
                ],
                'metadata': {'#' + col: str(row[col]) for col in manifest_columns},
                'id_field': id_field,
                'old_subject_id': uploaded_id_map.get(id_field) if is_replacement else None,
            }
            manifest.append(entry)

        print('Manifest created')

        untracked_replacements = [
            item['id_field'] for item in manifest
            if item['id_field'] in replace_ids and item['id_field'] in uploaded_id_map and item['old_subject_id'] is None
        ]
        if untracked_replacements:
            print(
                f"Note: {len(untracked_replacements)} replacement(s) have no recorded subject id "
                f"(uploaded before ID tracking was added), so the old subject can't be auto-removed: "
                f"{untracked_replacements}"
            )

        pbar = tqdm(total=len(manifest), unit=' subjects uploaded')
        pbar_lock = threading.Lock()

        # Split manifest into batches
        for batch_start in range(0, len(manifest), batch_size):
            batch = manifest[batch_start:batch_start + batch_size]

            # Prepare argument tuples for starmap
            batch_args = [(item, gz_project_id, client, pbar, pbar_lock) for item in batch]

            pool = ThreadPool(n_processes)
            new_subjects = pool.starmap(save_subject_wrapper, batch_args)
            pool.close()
            pool.join()


            with client:
                try:
                    subject_set.add(new_subjects)
                except:
                    subject_set.reload()
                    subject_set.add(new_subjects)

                # Remove the old, incorrect subjects being replaced in this batch
                old_ids_to_remove = [item['old_subject_id'] for item in batch if item['old_subject_id']]
                if old_ids_to_remove:
                    try:
                        old_subjects = [Subject.find(old_id) for old_id in old_ids_to_remove]
                        subject_set.remove(old_subjects)
                    except Exception as e:
                        print(f"Warning: failed to remove old subject(s) {old_ids_to_remove}: {e}")

            # Now that they're added (and replaced), update the txt file for each id
            for item, new_subject in zip(batch, new_subjects):
                update_txt_file(item['id_field'], new_subject.id)

        pbar.close()
        print("All uploads completed.")
    finally:
        # Release the pooled HTTP connections so repeated runs in the same
        # kernel don't leak sockets/threads (was causing kernel restart hangs).
        client.session.close()

def replace_selected_galaxies(selected_galaxies, subject_set_id, subject_png_save_dir, gz_project_id, manifest_columns, n_processes=10, batch_size=10):
    """
    Re-upload corrected galaxies, replacing any previously-uploaded subject that shares
    the same cutout_stem (the old subject is removed from the subject set once its
    replacement is added).
    """
    ids = selected_galaxies['cutout_stem'].astype(str).tolist()
    upload_selected_galaxies(
        selected_galaxies,
        subject_set_id=subject_set_id,
        subject_png_save_dir=subject_png_save_dir,
        gz_project_id=gz_project_id,
        manifest_columns=manifest_columns,
        n_processes=n_processes,
        batch_size=batch_size,
        replace_ids=ids,
    )

def save_subject_wrapper(manifest_item, gz_project_id, client, pbar=None, pbar_lock=None):
    """
    Upload a single subject (two images) to Galaxy Zoo.
    """
    subject = Subject()
    subject.links.project = gz_project_id

    for img_loc in manifest_item['jpg_locs']:
        assert os.path.exists(img_loc), f"Image not found: {img_loc}"
        subject.add_location(img_loc)

    subject.metadata.update(manifest_item['metadata'])
    subject.save(client=client)

    if pbar and pbar_lock:
        with pbar_lock:
            pbar.update()

    return subject


In [21]:
upload_selected_galaxies(
    csv,
    subject_set_id=subject_set_id,  
    subject_png_save_dir='/users/7/aimees/AI_Inspector/2681/11/', #'/users/7/aimees/AI_Inspector/betatest/',
    gz_project_id=gz_project_id,
    manifest_columns=manifest_columns,
    n_processes=10,
    batch_size=100,
)

Manifest created


0 subjects uploaded [00:00, ? subjects uploaded/s]

All uploads completed.


### Re-upload corrected subjects (same id, replacing the old one)

`replace_selected_galaxies` treats every row in `corrected_csv` as a replacement: it uploads
the corrected image(s) even though `cutout_stem` was already uploaded, then removes the old
subject from `subject_set` (if that subject's id was recorded when it was originally uploaded).

In [22]:
# corrected_csv = pd.read_csv('/path/to/corrected_detections.csv')
# corrected_csv['Index'] = range(0, len(corrected_csv))
#
# replace_selected_galaxies(
#     corrected_csv,
#     subject_set_id=subject_set_id,
#     subject_png_save_dir='/users/7/aimees/AI_Inspector/2681/11',
#     gz_project_id=gz_project_id,
#     manifest_columns=manifest_columns,
#     n_processes=10,
#     batch_size=100,
# )

In [32]:
corrected_csv = pd.read_csv('/users/7/aimees/AI_Inspector/betatest/beta.csv')
corrected_csv['Index'] = range(0, len(corrected_csv))

replace_selected_galaxies(
    corrected_csv,
    subject_set_id=subject_set_id,
    subject_png_save_dir='/users/7/aimees/AI_Inspector/betatest/',
    gz_project_id=gz_project_id,
    manifest_columns=manifest_columns,
    n_processes=10,
    batch_size=100,
)

Manifest created


100%|██████████| 150/150 [00:43<00:00,  3.46 subjects uploaded/s]

All uploads completed.


### Clean up leftover duplicate beta subjects

The 150 subjects above were freshly uploaded into the Beta Test set, but the 150
subjects originally *linked* into it from the main cutouts set (137608, via
`BetaDataset.ipynb`) were never removed first. Since those old subjects are the
exact same `Subject` objects as in set 137608, we can identify them by ID overlap.

In [34]:
# Diagnostic only (no changes made). Find subjects in the Beta Test set that are
# duplicates: same Subject linked from the main cutouts set (137608), vs. the 150
# freshly re-uploaded just above.
beta_source_set_id = 138337  # '2681_11832_3_DET11 - cutouts'
beta_set = SubjectSet.find(subject_set_id)  # 137684, Beta Test

source_ids = {s.id for s in SubjectSet.find(beta_source_set_id).subjects}
beta_subjects = list(beta_set.subjects)

old_linked_subjects = [s for s in beta_subjects if s.id in source_ids]
new_uploaded_subjects = [s for s in beta_subjects if s.id not in source_ids]

print(f"Beta Test set has {len(beta_subjects)} subjects total")
print(f"  {len(old_linked_subjects)} linked from the main cutouts set (candidates to remove)")
print(f"  {len(new_uploaded_subjects)} freshly re-uploaded just now (keep these)")

Beta Test set has 300 subjects total
  0 linked from the main cutouts set (candidates to remove)
  300 freshly re-uploaded just now (keep these)


In [ ]:
# Only run after checking the counts printed above look right (should be ~150 old,
# ~150 new). This unlinks the old subjects from the Beta Test set — they remain
# untouched in the main cutouts set (137608), so nothing is deleted from Zooniverse,
# just removed from Beta Test.
beta_set.remove(old_linked_subjects)
print(f"Removed {len(old_linked_subjects)} old subjects from Beta Test set {subject_set_id}")

# Upload Full Panel

In [5]:
# Upload a single full-detector image (not a cutout) as its own new subject set
detector_image_path = '/users/7/aimees/AI_Inspector/2681/11/2681_11832_3_DET11.png'
assert os.path.exists(detector_image_path), f"Image not found: {detector_image_path}"

client = Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))
try:
    with client:
        project = Project.find(gz_project_id)

        detector_subject_set = SubjectSet()
        detector_subject_set.links.project = project
        detector_subject_set.display_name = '2681_11832_3_DET11 - full detector'
        detector_subject_set.save()

        subject = Subject()
        subject.links.project = gz_project_id
        subject.add_location(detector_image_path)
        subject.metadata.update({
            '#field': '2681',
            '#detector': 'DET11',
            '#source_fits': '2681_11832_3',
        })
        subject.save(client=client)

        detector_subject_set.add(subject)
    print(f"Uploaded {detector_image_path} as subject {subject.id} to subject set {detector_subject_set.id} ({detector_subject_set.display_name})")
finally:
    client.session.close()


PanoptesAPIException: Validation failed: Display name has already been taken